In [ ]:
!pip install git+https://github.com/facebookresearch/segment-anything.git
!pip install opencv-python matplotlib
!wget https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth -O sam_vit_b.pth


In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import torch
from segment_anything import sam_model_registry, SamAutomaticMaskGenerator

# Load the image
image_path = "/kaggle/input/sample-image/test.jpg"
image = cv2.imread(image_path)
image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

# Resize the image if needed
desired_width, desired_height = 512, 512
image = cv2.resize(image, (desired_width, desired_height))

# Convert to grayscale
gray_image = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)

# Apply Gaussian Blur
gaussian_blurred = cv2.GaussianBlur(gray_image, (5, 5), 0)

# Apply Roberts Cross Operator
roberts_cross_x = np.array([[1, 0], [0, -1]], dtype=np.float32)
roberts_cross_y = np.array([[0, 1], [-1, 0]], dtype=np.float32)
roberts_x = cv2.filter2D(gaussian_blurred, cv2.CV_64F, roberts_cross_x)
roberts_y = cv2.filter2D(gaussian_blurred, cv2.CV_64F, roberts_cross_y)
roberts_edges = cv2.magnitude(roberts_x, roberts_y)

# Convert to 8-bit
roberts_edges_8bit = cv2.convertScaleAbs(roberts_edges)

# Load SAM
sam = sam_model_registry["vit_b"](checkpoint="sam_vit_b.pth")
sam.to("cuda" if torch.cuda.is_available() else "cpu")

# Initialize mask generator
mask_generator = SamAutomaticMaskGenerator(
    model=sam,
    points_per_side=64,
    pred_iou_thresh=0.8,
    stability_score_thresh=0.85,
    crop_n_layers=1,
    crop_n_points_downscale_factor=2,
    min_mask_region_area=70
)

# Convert sharpened edges to 3-channel
roberts_edges_3d = np.stack([roberts_edges_8bit]*3, axis=-1)

# Generate masks
masks = mask_generator.generate(roberts_edges_3d)

# Post-process masks
def post_process_masks(masks, min_area=500):
    processed_masks = []
    for mask in masks:
        if np.sum(mask['segmentation']) > min_area:
            processed_masks.append(mask)
    return processed_masks

masks = post_process_masks(masks)

# Visualization function
def show_anns(masks):
    if len(masks) == 0:
        return
    sorted_anns = sorted(masks, key=lambda x: x['area'], reverse=True)
    ax = plt.gca()
    ax.set_autoscale_on(False)
    img = np.ones((sorted_anns[0]['segmentation'].shape[0],
                   sorted_anns[0]['segmentation'].shape[1], 4))
    for ann in sorted_anns:
        mask = ann['segmentation']
        color_mask = np.concatenate([np.random.random(3), [0.35]])
        img[mask == 1] = color_mask
    ax.imshow(img)

# Plot results
plt.figure(figsize=(12, 12))

plt.subplot(2, 3, 1)
plt.imshow(image)
plt.title("Original Image")
plt.axis("off")

plt.subplot(2, 3, 2)
plt.imshow(gray_image, cmap="gray")
plt.title("Grayscale Image")
plt.axis("off")

plt.subplot(2, 3, 3)
plt.imshow(gaussian_blurred, cmap="gray")
plt.title("Gaussian Blurred")
plt.axis("off")

plt.subplot(2, 3, 4)
plt.imshow(roberts_edges_8bit, cmap="gray")
plt.title("Roberts Edge Detection")
plt.axis("off")

plt.subplot(2, 3, 5)
plt.imshow(roberts_edges_8bit, cmap="gray")
plt.title("Edge Detection (Roberts)")
plt.axis("off")

plt.subplot(2, 3, 6)
plt.imshow(roberts_edges_8bit, cmap="gray")
show_anns(masks)
plt.title("SAM Segmentation (Roberts)")
plt.axis("off")

plt.tight_layout()
plt.show()


In [ ]:
from sklearn.cluster import DBSCAN
import os
def cluster_rows(masks, eps=15, min_samples=1):
    """
    Cluster masks into rows using DBSCAN on vertical coordinate (bbox top y).
    eps controls max vertical distance between items in same cluster (row).
    """
    ys = np.array([mask['bbox'][1] for mask in masks]).reshape(-1, 1)
    clustering = DBSCAN(eps=eps, min_samples=min_samples).fit(ys)
    labels = clustering.labels_

    unique_labels = sorted(set(labels))
    rows = []
    for label in unique_labels:
        row_masks = [masks[i] for i in range(len(masks)) if labels[i] == label]
        # Sort this row left to right
        row_masks = sorted(row_masks, key=lambda m: m['bbox'][0])
        rows.append(row_masks)

    # Sort rows top to bottom by mean y position
    rows = sorted(rows, key=lambda row: np.mean([m['bbox'][1] for m in row]))

    # Flatten the list
    return [mask for row in rows for mask in row]

sorted_masks = cluster_rows(masks, eps=20)  # eps can be adjusted per use case/image

# Directory to save extracted shapes
output_dir = "extracted_shapes"
os.makedirs(output_dir, exist_ok=True)

# Extract and save/display each shape as RGBA with transparency
for idx, mask_info in enumerate(sorted_masks):
    mask = mask_info['segmentation']
    
    extracted_rgba = np.zeros((image.shape[0], image.shape[1], 4), dtype=np.uint8)
    extracted_rgba[mask, :3] = image[mask]  # RGB channels
    extracted_rgba[mask, 3] = 255            # Alpha channel
    
    save_path = os.path.join(output_dir, f"shape_rowwise_clustered_{idx+1}.png")
    cv2.imwrite(save_path, cv2.cvtColor(extracted_rgba, cv2.COLOR_RGBA2BGRA))
    
    plt.figure(figsize=(4, 4))
    plt.imshow(extracted_rgba)
    plt.axis('off')
    plt.title(f"Extracted Shape {idx + 1} (Clustered row-wise)")
    plt.show()

In [ ]:
import cv2
import numpy as np
import os

# Desired high resolution for resizing (width, height)
desired_width, desired_height = 256, 256  # You can increase this further for higher resolution

# Directory to save cropped and resized symbols
output_dir = "cropped_resized_symbols_high_quality"
os.makedirs(output_dir, exist_ok=True)

# Loop through each mask
for idx, mask_info in enumerate(sorted_masks):
    mask = mask_info['segmentation']
    
    # Get the bounding box for the mask and convert to integers
    x, y, w, h = map(int, mask_info['bbox'])  # Ensure the bbox values are integers
    
    # Crop the symbol using the bounding box (retain high resolution)
    cropped_symbol = image[y:y+h, x:x+w]
    
    # Resize the cropped symbol to the desired size using high-quality interpolation methods
    resized_symbol = cv2.resize(cropped_symbol, (desired_width, desired_height), interpolation=cv2.INTER_CUBIC)
    
    # Save the cropped and resized symbol
    save_path = os.path.join(output_dir, f"cropped_resized_symbol_{idx+1}.png")
    cv2.imwrite(save_path, cv2.cvtColor(resized_symbol, cv2.COLOR_RGB2BGR))
    
    # Display the cropped and resized symbol
    plt.figure(figsize=(4, 4))
    plt.imshow(resized_symbol)
    plt.axis('off')
    plt.title(f"Cropped & Resized Symbol {idx + 1}")
    plt.show()


In [ ]:
import cv2
import numpy as np
import os

# Directory to save cropped symbols (no resizing)
output_dir = "cropped_symbols_high_quality"
os.makedirs(output_dir, exist_ok=True)

# Loop through each mask
for idx, mask_info in enumerate(sorted_masks):
    mask = mask_info['segmentation']
    
    # Get the bounding box for the mask and convert to integers
    x, y, w, h = map(int, mask_info['bbox'])  # Ensure the bbox values are integers
    
    # Crop the symbol using the bounding box (retain high resolution)
    cropped_symbol = image[y:y+h, x:x+w]
    
    # Save the cropped symbol (without resizing)
    save_path = os.path.join(output_dir, f"cropped_symbol_{idx+1}.png")
    cv2.imwrite(save_path, cv2.cvtColor(cropped_symbol, cv2.COLOR_RGB2BGR))
    
    # Display the cropped symbol
    plt.figure(figsize=(4, 4))
    plt.imshow(cropped_symbol)
    plt.axis('off')
    plt.title(f"Cropped Symbol {idx + 1} (High Quality)")
    plt.show()


In [ ]:
import pandas as pd

# Load the Excel file
df = pd.read_excel('/kaggle/input/gardiners-list-csv/Alan Gardiners List of Hieroglyphic Signs.xlsx')

# Check the dataframe structure after loading
print(df.head())
